In [2]:
%load_ext autoreload
%autoreload 2

from src.preprocessing import *
import src.ocp as ocp

from src.trading import *

from pathlib import Path
import polars as pl
import numpy as np

In [3]:
#1) Get selected tickers
tickers = get_selected_tickers()

# 2) Load per-ticker returns
frames = load_selected_ticker_frames()

2026-01-21 15:31:31,435 | INFO | Found 91 selected tickers.
2026-01-21 15:31:32,266 | INFO | Loaded 90 selected ticker frames into memory.


In [4]:
DATA_DIR = Path("data/selected/SP100/bbo")
TICKERS = tickers

daily_matrices = ocp.build_daily_return_matrices(
    data_dir=DATA_DIR,
    ticker_list=TICKERS,
    return_col="mid_price_return"
)

print(f"Built {len(daily_matrices)} daily return matrices.")

daily_pairs = ocp.build_daily_pairs(daily_matrices)


Built 565 daily return matrices.


In [5]:
daily_matrices

{datetime.date(2015, 1, 2):                             AAPL.OQ     ABT.N     ACN.N     AIG.N     ALL.N  \
 timestamp                                                                     
 2015-01-02 09:32:00-05:00  0.000404 -0.000443  0.000000 -0.000796 -0.001696   
 2015-01-02 09:33:00-05:00 -0.000045 -0.000775  0.000335 -0.000266  0.000991   
 2015-01-02 09:34:00-05:00 -0.001033  0.002881  0.001450  0.001772  0.002122   
 2015-01-02 09:35:00-05:00 -0.001124 -0.000331  0.001782 -0.001238  0.000565   
 2015-01-02 09:36:00-05:00  0.001531  0.003427  0.000056 -0.003276 -0.002187   
 ...                             ...       ...       ...       ...       ...   
 2015-01-02 15:56:00-05:00 -0.000274  0.000000 -0.000056  0.000534  0.000214   
 2015-01-02 15:57:00-05:00  0.000183  0.000334 -0.000561  0.000000  0.000143   
 2015-01-02 15:58:00-05:00 -0.001188 -0.000556 -0.001123 -0.000979 -0.001070   
 2015-01-02 15:59:00-05:00  0.000183  0.000445 -0.000112  0.000089  0.000500   
 2015-01-02 1

In [6]:
first_n_days = 573
daily_top = ocp.ocp_run_all_days_fast(
    daily_matrices=daily_matrices,
    daily_pairs=daily_pairs,
    top_k=10,
    keep_top_pairs=None,
    max_lag=50,       
    band=10,          
    first_n_days=first_n_days
)

daily_top.to_parquet(f"data/top_pairs/daily_top_pairs_{first_n_days}.parquet", index=False)

 45%|████▌     | 255/565 [03:35<05:09,  1.00it/s]c:\Users\wahid\anaconda3\envs\ADA\Lib\site-packages\numpy\_core\_methods.py:190: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
c:\Users\wahid\Desktop\FBD\OCP-StatArb-SP100\src\ocp.py:494: RuntimeWarning: invalid value encountered in subtract
  return (z - mu) / (s + 1e-12)
 71%|███████   | 400/565 [05:44<02:11,  1.26it/s]c:\Users\wahid\anaconda3\envs\ADA\Lib\site-packages\numpy\_core\_methods.py:190: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
c:\Users\wahid\Desktop\FBD\OCP-StatArb-SP100\src\ocp.py:494: RuntimeWarning: invalid value encountered in subtract
  return (z - mu) / (s + 1e-12)
 80%|████████  | 454/565 [06:30<01:34,  1.18it/s]c:\Users\wahid\anaconda3\envs\ADA\Lib\site-packages\numpy\_core\_methods.py:190: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
c:\Users\wahid\Desktop\FBD\OCP-StatArb-SP100\src\ocp.py:49

In [7]:
daily_top = pd.read_parquet(f"data/top_pairs/daily_top_pairs_573.parquet")

daily_top

,date,leader,follower,l_hat,sigma_l,cost
0,2015-01-02,HON.N,V.N,1.010965,1.614725,201.883981
1,2015-01-02,WFC.N,KO.N,1.129870,1.680167,210.109826
2,2015-01-02,ALL.N,KO.N,1.121739,1.862200,208.889776
3,2015-01-02,UNP.N,NEE.N,1.025424,1.925352,218.139694
4,2015-01-02,MMM.N,GD.N,1.043860,1.941650,200.560815
...,...,...,...,...,...,...
5645,2017-03-31,BAC.N,WFC.N,1.009070,2.429955,171.826367
5646,2017-03-31,JPM.N,V.N,1.184486,2.434795,219.740487
5647,2017-03-31,EMR.N,WFC.N,1.612903,2.464510,212.564136
5648,2017-03-31,AIG.N,MS.N,1.183158,2.511894,207.897267


In [34]:
PAIRS_PATH = Path("data/top_pairs/daily_top_pairs_573_90.parquet")
RETURNS_DIR = Path("data/selected/SP100/bbo")
INDEX_TICKER = "SPY.P"

params = OCPTradeParams(
    lookback=20,
    k=2.5,
    tc_bps=2.0,
    r_bps=4.0,
    max_lag_minutes=30,
    ci_z=2.8,
    enter_on_next_bar=True,
)

In [35]:
results = run_backtest(
    pairs_path=PAIRS_PATH,
    returns_dir=RETURNS_DIR,
    index_ticker=INDEX_TICKER,
    params=params,
)

In [36]:
results

trade_day,daily_pnl,pairs_used,entries,exits
date,f64,i64,i64,i64
2015-01-05,0.000554,5,20,20
2015-01-06,-0.016512,8,36,36
2015-01-07,-0.005762,4,13,13
2015-01-08,0.001664,5,26,26
2015-01-09,0.000025,2,8,8
…,…,…,…,…
2017-04-06,0.0,0,0,0
2017-04-07,0.0,0,0,0
2017-04-10,0.0,0,0,0


In [37]:
daily = results["daily_pnl"].to_numpy()
mu = daily.mean()
sigma = daily.std(ddof=0)
sharpe = (mu / sigma) * np.sqrt(252) if sigma > 0 else np.nan
print("Approx Sharpe:", sharpe)
print("Total PnL:", daily.sum())
print("Mean Daily PnL:", mu)

Approx Sharpe: 0.11441079053377863
Total PnL: 0.3446949276750766
Mean Daily PnL: 0.0006026135099214626


In [31]:
df = results.to_pandas()